# 🗓 Calendar Agent — Colab LoRA Training

Kaggle GPU quota 소진 시 대안. Kaggle 노트북(`calendar_kaggle.ipynb`)과 **동일한 학습 파이프라인**을 Colab에서 실행한다.

## Kaggle vs Colab 핵심 차이

| 항목 | Kaggle | Colab |
|---|---|---|---|
| 작업 경로 | `/kaggle/working/` | `/content/` |
| GPU 수 | T4 × 2 (DDP) | T4 × 1 (무료) / A100 × 1 (Pro) |
| 학습 명령 | `torchrun --nproc_per_node=2` | `python` 단독 |
| 정밀도 | bf16 | fp16 (무료 T4) / bf16 (Pro A100·V100) |
| 결과 다운로드 | Output 패널 또는 Kaggle API | `files.download()` 또는 Drive 마운트 |
| RAPIDS 충돌 | 있음 (grep으로 숨김) | 없음 |

## 실행 전 준비 (한 번만)

### 1. Colab 런타임 설정
- 상단 메뉴 → `런타임` → `런타임 유형 변경`
- **하드웨어 가속기**: `T4 GPU` (무료) 또는 `A100 GPU` (Pro 권장)
- `저장` 클릭

### 2. GitHub PAT 발급 (Private repo clone 용)
https://github.com/settings/tokens/new?type=classic — repo scope, 7 days

---

준비 끝나면 셀을 위에서 아래로 순서대로 실행.  
예상 시간: T4 단독 약 **2시간**, Pro A100 약 **45분**.

## 0. GPU 확인

`Tesla T4` 또는 `A100-SXM4` 표시되어야 함.  
안 보이면 `런타임 → 런타임 유형 변경`에서 GPU 선택 후 재연결.

In [ ]:
!nvidia-smi

In [ ]:
import torch
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
SUPPORTS_BF16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
print(f'GPU: {GPU_NAME}')
print(f'bf16 지원: {SUPPORTS_BF16}')
print(f'  → 권장 config: {"train.yaml (bf16)" if SUPPORTS_BF16 else "train_colab.yaml (fp16)"}')

## 1. 레포 clone (Private)

Colab 작업 디렉토리는 `/content/`. 그 안에 clone.

In [ ]:
import os, getpass

%cd /content
if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/sooryong/calendar-agent.git
    token = None
else:
    print('calendar-agent already cloned, force-syncing to origin/main…')
    !cd calendar-agent && git fetch origin && git reset --hard origin/main

%cd /content/calendar-agent
!git log --oneline -1

## 2. 학습 의존성 설치

Kaggle과 달리 RAPIDS 사전설치가 없어서 충돌 경고 없음.

In [ ]:
!pip install -q -e .[train]
!pip uninstall torchao -y -q 2>/dev/null || true   # PEFT가 거부하는 구버전 torchao 제거 (없으면 무해)

import os, torch
os.environ["WANDB_DISABLED"] = "true"
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "ngpu:", torch.cuda.device_count())

## 3. 데이터 확인 (repo에 포함 — clone으로 따라옴)

Kaggle 데이터셋 첨부 불필요. git이 단일 출처.

In [ ]:
for p in ["data/processed/train.jsonl", "data/processed/val.jsonl", "data/eval/real_golden.jsonl"]:
    n = sum(1 for _ in open(p, encoding="utf-8"))
    print(f"{p}: {n} records")

## 4. 학습 Config 선택

| GPU | 권장 config | 정밀도 | 예상 시간 |
|---|---|---|---|
| T4 (무료) | `configs/train_colab.yaml` | fp16 | ~2시간 |
| A100 / V100 (Pro) | `configs/train.yaml` | bf16 | ~45분 |

> 위 0번 셀이 `bf16 지원: True`로 표시되면 `configs/train.yaml` 사용 권장.

In [ ]:
# T4는 bf16 tensor core가 없지만 PyTorch 소프트웨어 에뮬로 bf16 실행 가능.
# Kaggle 노트북도 T4×2에서 bf16으로 학습 중 (r11=0.905).
# → T4에서도 bf16(train.yaml) 사용 권장. fp16은 학습 안정성이 낮아 성능 저조 가능.
# 단, bf16은 T4에서 fp16보다 step당 ~20~30% 느림 (tensor core 미사용).

CONFIG = 'configs/train.yaml'   # bf16 — T4 포함 모든 GPU 권장

# fp16으로 바꾸려면 아래 줄 주석 해제 (비권장: 성능 저조 이력 있음)
# CONFIG = 'configs/train_colab.yaml'   # fp16

import torch
print('학습 config:', CONFIG)
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'bf16 tensor core 지원: {torch.cuda.is_bf16_supported()} (False여도 bf16 실행 가능)')

## 5. 학습 실행

약 2245건 × 3 epochs ≈ 192 step.  
Colab은 GPU 1개이므로 `torchrun` 대신 `python` 단독 실행.

> ⚠️ Colab 무료 세션은 **12시간** 한도, 유휴 시 **90분** 자동 종료.  
> T4 단독 학습(~2h)은 한도 내이나, 브라우저 탭을 유지할 것.

In [ ]:
# Colab = 단일 GPU → torchrun 없이 python 직접 실행 (Kaggle T4x2 DDP와 달리 단일 프로세스)
!python scripts/train_lora.py --config {CONFIG}

## 5.5. 학습 직후 평가 (선택)

merge 후 골든셋으로 바로 평가해 `time_match_rate` / `final_score` 확인.  
로컬 CPU 평가(약 14분)보다 GPU가 훨씬 빠름. 결과 json은 아래 6번 zip에 포함됨.

In [ ]:
import yaml, os
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
BASE       = _mcfg['hf_id']                        # Qwen2.5-0.5B-Instruct 등 자동
LORA_DIR   = _cfg['output_dir']                    # 예: models/lora/r14-qwen0.5b
NAME       = os.path.basename(LORA_DIR)            # r14-qwen0.5b
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON  = f'logs/eval_{NAME}.json'
GOLDEN     = _cfg.get('eval_golden', 'data/eval/real_golden.jsonl')
ZIP        = f'/content/lora_{NAME}.zip'
print(f'base={BASE}\nlora={LORA_DIR}\ngolden={GOLDEN}\nzip={ZIP}')

# merge → 골든셋 평가
!python scripts/merge_lora.py --base {BASE} --lora {LORA_DIR} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval {GOLDEN} --out {EVAL_JSON}
print(f'--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

## 6. 결과 패키징 & 다운로드

Colab에서 파일을 로컬로 가져오는 방법 3가지:

1. **`files.download()`** (아래 셀) — 브라우저 직접 다운로드. 1GB 이하 권장.
2. **Google Drive 마운트** → zip을 Drive에 복사 (큰 파일 또는 세션 만료 대비)
3. 좌측 파일 패널 → zip 우클릭 → 다운로드

> ⚠️ Colab 세션 종료 시 `/content/` 내용이 **사라짐**. 반드시 세션 중에 다운로드 또는 Drive에 저장할 것.

In [ ]:
!ls -la {LORA_DIR}/
!zip -r {ZIP} {LORA_DIR} {CONFIG} configs/lora.yaml {_cfg['model_config']} {EVAL_JSON}
!ls -lh {ZIP}

In [ ]:
# 방법 1: 브라우저 직접 다운로드
from google.colab import files
files.download(ZIP)

In [ ]:
# 방법 2: Google Drive에 저장 (세션 만료 대비 — 큰 파일 또는 장기 보관 시 권장)
# 아래 셀 실행 시 Drive 마운트 인증 팝업이 뜸
from google.colab import drive
drive.mount('/drive')

import shutil, os
DRIVE_DIR = '/drive/MyDrive/calendar-agent'
os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copy(ZIP, DRIVE_DIR)
shutil.copy(EVAL_JSON, DRIVE_DIR)
print(f'Drive에 저장 완료: {DRIVE_DIR}')
!ls -lh {DRIVE_DIR}

## 7. 로컬에서 이어서 작업 (merge → quantize)

zip을 로컬 `D:\calendar-agent\`에 압축 해제한 뒤:

```powershell
# zip 압축 해제 (lora 어댑터가 models/lora/<NAME>/ 에 들어옴)
Expand-Archive lora_<NAME>.zip -DestinationPath D:\calendar-agent\ -Force

# merge (로컬 — CPU도 가능, 단 느림)
python scripts/merge_lora.py `
    --base Qwen/Qwen2.5-0.5B-Instruct `
    --lora models/lora/<NAME> `
    --out  models/merged/<NAME>

# 양자화 (llama.cpp 빌드 필요 — SETUP.md 참고)
bash scripts/quantize.sh models/merged/<NAME> models/gguf/<NAME>
```

> 평가(eval_model.py)를 Colab에서 이미 돌렸다면 `logs/eval_<NAME>.json`이 zip에 포함됨.  
> 로컬에서 추가 평가가 필요하면 같은 명령으로 재실행 가능 (CPU라 약 14분 소요).